# 🧮 groupby — split, summarise, combine

**Time:** ~35 minutes &nbsp;•&nbsp; **Goal:** answer "which X has the most Y" for any X and any Y in this table, in one line.

Every question worth asking about this survey has the same shape:

> *Per allotment* — how many bumblebees?
> *Per crop* — how many honeybees?
> *Per month, per year* — how much surveying was done?

All of them are one operation: **split** the table into groups, **summarise** each group, **combine** the answers back into a table. pandas calls it `groupby`, and once you can see that shape in a question you can write the line without thinking.

## Run this first

In [ ]:
# Run this first. Everything sessions 1 and 2 taught you, in one cell.
import urllib.request
from pathlib import Path

import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/HocheggerLab/y3-bio-python/main/lab06/data/"

SURVEY = "pollinators.csv"
if not Path(SURVEY).exists():
    urllib.request.urlretrieve(DATA_URL + SURVEY, SURVEY)

df = pd.read_csv(SURVEY)

df.columns = df.columns.str.strip()                      # the header bug
df = df.rename(columns={"Crop_Generic": "Crop", "Total_Flowers": "Flowers"})
df["Date"] = pd.to_datetime(df["Date"])                  # real dates
df["Month"] = pd.Categorical(                            # months in season order
    df["Date"].dt.month_name(),
    categories=["April", "May", "June", "July", "August", "September"],
    ordered=True,
)
df["Crop"] = df["Crop"].replace({"Rbean": "Runner bean", "Bbean": "Broad bean"})

INSECTS = ["Beetles", "Hoverflies", "Flies", "Butterflies",
           "Honeybees", "Solitary_Bees", "Wasps", "Bumblebees"]
df["All_Insects"] = df[INSECTS].sum(axis=1)

print(df.shape)
df.head(3)

## Skill 1 — One key, one column, one answer each

```python
df.groupby("Allotment")["Bumblebees"].sum()
```

Read it left to right, and say it out loud:

- `df.groupby("Allotment")` — split the table into nine piles, one per allotment
- `["Bumblebees"]` — from each pile, take this column
- `.sum()` — and add it up

What comes back is a **Series**: the index is the group names, the values are the answers. Which means everything you know about a Series still applies — `.sort_values()`, `.nlargest()`, `.idxmax()`.

In [ ]:
# DEMO - bumblebees per allotment
per_site = df.groupby("Allotment")["Bumblebees"].sum()
print(per_site.sort_values(ascending=False))

print()
print("busiest site:", per_site.idxmax(), "with", per_site.max(), "visits")

In [ ]:
# DEMO - the same line, three different summaries
print("--- total ---")
print(df.groupby("Crop")["Bumblebees"].sum().nlargest(5))

print()
print("--- mean per watch ---")
print(df.groupby("Crop")["Bumblebees"].mean().nlargest(5).round(2))

print()
print("--- how many watches ---")
print(df.groupby("Crop")["Bumblebees"].size().nlargest(5))

Three different top-fives from the same grouping. Already the shape of the problem is showing: **Courgette** climbs the mean table on very few watches, **Raspberry** dominates the totals because it was watched 269 times.

Hold that thought. The next notebook is about nothing else.

### 🧮 DIY challenge

1. Total **honeybee** visits per allotment. Which site tops it, and is it the same as for bumblebees?
2. Mean `Flowers` per crop — which crop puts up the most flowers per watch?
3. `df.groupby("Month", observed=True)["All_Insects"].sum()` — the busiest month. Is it in calendar order? (It will be, because you made `Month` an ordered Categorical in session 2.)

> 💡 `observed=True` tells pandas not to invent rows for categories that never appear. Leave it off once and you will see why it exists.

In [ ]:
# TODO - your turn

## Skill 2 — Several summaries at once, with `.agg()`

One number per group is rarely enough. You almost always want the total *and* the mean *and* how many rows it came from — because a mean over three watches is not the same kind of fact as a mean over three hundred.

```python
df.groupby("Crop")["Bumblebees"].agg(["sum", "mean", "count"])
```

Better still, **named aggregation**: you choose the column names, and you can pull from different columns in the same call.

```python
df.groupby("Crop").agg(
    watches=("Bumblebees", "size"),
    bumblebees=("Bumblebees", "sum"),
    flowers=("Flowers", "sum"),
)
```

Each argument reads `new_name=(column, what_to_do)`. This is the form worth memorising — it produces a table you can hand to somebody without explaining it.

In [ ]:
# DEMO - a summary table you could put in a report
summary = df.groupby("Crop").agg(
    watches=("Bumblebees", "size"),
    bumblebees=("Bumblebees", "sum"),
    honeybees=("Honeybees", "sum"),
    flowers=("Flowers", "sum"),
)

summary.sort_values("bumblebees", ascending=False)

### 🧮 DIY challenge

1. Build the same summary per **allotment** instead of per crop.
2. Add a `mean_flowers=("Flowers", "mean")` column and round it to one decimal place.
3. `summary` is a DataFrame with `Crop` as its index. Call `.reset_index()` on it — what changes? Why would you want that before saving or plotting?

> 💡 After a `groupby`, the grouping key is the **index**, not a column. `.reset_index()` turns it back into an ordinary column. You will use it constantly.

In [ ]:
# TODO - your turn

## Skill 3 — Two keys at once

Pass a **list** of columns and pandas splits by every combination.

```python
df.groupby(["Year", "Month"], observed=True)["Bumblebees"].sum()
```

The result has a **MultiIndex** — two levels of labels down the side. It is the honest shape for the answer, and it is awkward to work with, so there are two ways to flatten it:

- `.reset_index()` turns both levels into ordinary columns — a long, tidy table
- `.unstack()` moves the *last* level up into columns — a wide, readable grid

In [ ]:
# DEMO - two keys, three shapes of the same answer
by_ym = df.groupby(["Year", "Month"], observed=True)["Bumblebees"].sum()

print("--- as a MultiIndex Series ---")
print(by_ym)

In [ ]:
# DEMO - flattened two ways
print("--- reset_index: long and tidy ---")
print(by_ym.reset_index().head())

print()
print("--- unstack: wide and readable ---")
print(by_ym.unstack())

The wide version is the one to look at; the long version is the one to compute with. Notice the `NaN` in the wide grid — 2017 has no April or July watches at all. `unstack` has to put *something* in those cells, and a missing value is the honest answer.

### 🧮 DIY challenge

1. Group by `Allotment` **and** `Community`. Which allotments have both kinds of plot?
2. Group by `Year` and `Crop` and total the flowers. Unstack it. Which crops were only surveyed in one of the two years?
3. Group by `Allotment` and `Plot` and count the watches. How many plots does Weald have?

> 💡 `.unstack()` is easy to over-use. If the result is 80 columns wide, you wanted `.reset_index()`.

In [ ]:
# TODO - your turn

## Skill 4 — Counting things properly

Three methods that look interchangeable and are not:

| method | counts |
|---|---|
| `.size()` | **rows** in the group, missing values included |
| `.count()` | **non-missing values**, per column |
| `.nunique()` | **distinct values** |

The difference between `.size()` and `.count()` is exactly your missing data — which is why running both is a quick way to find it.

In [ ]:
# DEMO - three counts, three different questions
g = df.groupby("Allotment")

counts = pd.DataFrame({
    "watches": g.size(),
    "plants_recorded": g["Plants"].count(),
    "distinct_plots": g["Plot"].nunique(),
    "distinct_crops": g["Crop"].nunique(),
    "survey_days": g["Date"].nunique(),
})

counts.sort_values("watches", ascending=False)

That table is worth more than it looks. `Camp_Site` has **two** watches. `Moulsecoomb_Est` has eight. Any per-allotment mean you calculate is going to include those, and they will bounce around wildly while Weald's 495 watches sit still.

Deciding what to do about that is the last skill in this notebook.

### 🧮 DIY challenge

1. Which allotment recorded plant counts least reliably? (Compare `watches` with `plants_recorded`.)
2. How many distinct crops were watched at Roedale, and how many at Racehill?
3. Build the same table grouped by `Crop` instead. Which crop was watched on the most separate days?

> 💡 `pd.DataFrame({...})` built from a dictionary of Series lines them up on their shared index automatically. That is the same alignment that makes column arithmetic work.

In [ ]:
# TODO - your turn

## Skill 5 — The group that is too small 🏆

Here is the whole problem, in one table:

```python
df.groupby("Crop")["Bumblebees"].mean().nlargest(5)
```

Sort any group-mean and the top of the list fills up with groups of one or two. Not because they are remarkable, but because a small sample is free to be extreme. This is the single most common way a beginner's analysis goes wrong, and it is invisible unless you print `n` next to the answer.

**The habit:** never report a group statistic without the group size beside it, and filter out groups too small to mean anything.

```python
big = summary[summary["watches"] >= 40]
```

Where you draw the line is a judgement, and it belongs in your methods section — not hidden in the code.

In [ ]:
# DEMO - the trap, and the guard
stats = df.groupby("Crop").agg(
    watches=("Bumblebees", "size"),
    mean_bb=("Bumblebees", "mean"),
)

print("--- unguarded: look at the watch counts ---")
print(stats.nlargest(5, "mean_bb").round(2))

print()
print("--- only crops watched at least 40 times ---")
print(stats[stats["watches"] >= 40].nlargest(5, "mean_bb").round(2))

### 🏆 DIY finale — a defensible summary table

Build one table, per crop, containing:

1. the number of watches
2. total flowers, total bumblebees, total honeybees
3. the mean bumblebees per watch, rounded to two decimal places
4. filtered to crops watched at least 40 times
5. sorted so the most bumblebee-heavy crop is at the top

Then answer, in a sentence each:

- Which crop gets the most bumblebee attention, and how confident are you?
- Which gets the most honeybee attention? Is it a different crop?
- One crop is near the top for bumblebees and near the bottom for honeybees, or the other way round. Which, and can you think of a reason? (Look up what the flower of that crop is shaped like.)

> 💡 You are ranking on `mean_bb` — visits per *watch*. In the next notebook you will rank on visits per *flower*, and some of these answers will change. That is not a mistake in either notebook; it is the point.

In [ ]:
# TODO - your turn

## 🔭 Explore further

- The group-by guide, "split-apply-combine" in full: <https://pandas.pydata.org/docs/user_guide/groupby.html>
- `df.groupby("Crop")["Bumblebees"].describe()` gives you eight summary statistics per group in one call.
- `.transform()` is groupby's other half: instead of one row per group, it gives you one row per **original row**, filled with its group's value. Useful for "how does this watch compare to its allotment's average?"

## Recap

You can now:

- split, summarise and combine in one line with `df.groupby(key)[column].sum()`
- build a report-ready table with named aggregation: `agg(name=(column, func))`
- group by two keys, and flatten the result with `.reset_index()` or `.unstack()`
- tell `.size()`, `.count()` and `.nunique()` apart, and use the gap between them
- **always print `n` beside a group statistic**, and filter out groups too small to trust

**Next:** `06_groupby_rates_and_plots` — why every answer in this notebook might be the wrong way round, and how to draw the right one.